In [20]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from dotenv import load_dotenv
import os

In [8]:
# Leer el archivo .env para obtener las rutas de los documentos y la base de datos
load_dotenv()
DOCS_DIR = os.getenv("DOCS_DIR")
DB_DIR = os.getenv("DB_DIR")

In [9]:
print(f"Documentos en: {DOCS_DIR}")
print(f"Base de datos en: {DB_DIR}")

Documentos en: .\\docs
Base de datos en: .\\db


## Parte 1: Procesamiento de Documentos

In [10]:
# Cargar los documentos PDF
loader = PyPDFDirectoryLoader(DOCS_DIR)
documents = loader.load()

In [11]:
# Crear chunks de los documentos con repetición de 150 caracteres para evitar perder contexto
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap = 150)
chunks = text_splitter.split_documents(documents)

In [12]:
# 3. Embeddings y ChromaDB
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")
vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=DB_DIR)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8793.00it/s]


In [13]:
# El motor de búsqueda (en la base de datos)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
# 4. El LLM y el Prompt
llm = ChatOllama(model="qwen3:0.6b", temperature=0.1)

In [21]:
# 5. Crear el Prompt y la Cadena (Chain)
system_prompt = (
    "Eres un asistente virtual experto en el proceso de matrícula de la PUCP.\n"
    "Responde la pregunta del estudiante usando solo el contexto provisto.\n"
    "Si no encuentras la respuesta en el contexto, di amablemente que no tienes esa información.\n\n"
    "Contexto:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("\n=== ¡SISTEMA RAG LISTO! ===")

# --- PRUEBA DEL CHATBOT ---
# Prueba con una pregunta que sepas que está dentro de tus PDFs de "AvisoInicioDeMatricula"
pregunta = "¿Cuándo inicia el proceso de matrícula para el ciclo 2026-1?"
print(f"\nPregunta del Estudiante: {pregunta}")

respuesta = rag_chain.invoke({"input": pregunta})
print(f"\nRespuesta del Chatbot:\n{respuesta['answer']}")


=== ¡SISTEMA RAG LISTO! ===

Pregunta del Estudiante: ¿Cuándo inicia el proceso de matrícula para el ciclo 2026-1?


ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it